# 06. Planning and delegation

## Problem

A comparison depends on independently verified restore and incident research. One specialist may time out; the parent must retain partial results without marking the comparison verified.

## Learning objectives

Represent dependencies; verify completion; validate structured handoffs; bound concurrency and total deadlines; propagate cancellation; compare delegation costs.

Run top to bottom in a fresh kernel. The lab uses actual model calls, tools and temporary local storage. Configure the services before execution. Every plotted value declares its measurement kind.

In [ ]:
import os
from pathlib import Path
from tempfile import TemporaryDirectory
from IPython.display import display
ROOT = next(path for path in [Path.cwd(), *Path.cwd().parents]
            if (path / "pyproject.toml").exists())
os.chdir(ROOT)
scratch = TemporaryDirectory(prefix="northstar-course-")
os.environ["MPLCONFIGDIR"] = str(Path(scratch.name) / "matplotlib")
os.environ["LANGSMITH_TRACING"] = "false"
os.environ["LANGCHAIN_TRACING_V2"] = "false"
import matplotlib
get_ipython().run_line_magic("matplotlib", "inline")
import matplotlib.pyplot as plt
from deep_agent_lab.settings import Settings
from deep_agent_lab.courseware import diagram, bars
settings = Settings( deep_lab_mode="live", langfuse_enabled=False,
                    deep_lab_state_dir=Path(scratch.name) / "state")
assert settings.deep_lab_mode == "live" and not settings.langfuse_enabled

## Concept explanation

## Plans are not evidence of completion

Your repository already demonstrates `write_todos` and specialist delegation.

A todo list is useful for coordination, but its status is not proof that a task succeeded.

```
{
  "task": "Verify emergency access requirements",
  "status": "completed"
}
```

This might indicate only that the model called a tool and marked the item complete.

A stronger plan attaches completion criteria and actual evidence.

```
{
  "task_id": "T3",
  "task": "Verify emergency access requirements",
  "status": "verified",
  "evidence_ids": ["ACC-2#p4"],
  "verification": "Authorized policy revision confirmed"
}
```

The difference is analogous to a job queue reporting "handler executed" versus a domain operation reporting "required database state confirmed."

## Plan the dependency graph

Tasks A, B, and C can be performed independently.

Tasks D and E depend on earlier findings.

The save operation cannot happen before human review.

This dependency ordering should exist outside the model's unverified prose.

## Implement a verifiable plan

This is a simplified research-task model. Not every task requires citation evidence: a formatting task may instead require a validated output artifact. In a general planner, completion criteria should be typed according to the task.

The important design decision is that a completion state cannot be set merely because the model requested it.

## Specialist agents and structured handoffs

Delegation becomes attractive when subtasks require independent investigation or would otherwise pollute the parent's context.

For Northstar, useful specialists include the existing `policy-researcher` and `gap-reviewer`.

A good specialist task is narrow:

> Find the currently applicable production restore approval rule. Return the approving role, an exact supporting excerpt, evidence ID, source revision, and any unresolved uncertainty.

A poor task is:

> Research everything about database operations.

The second task has no clear completion condition and encourages unnecessary browsing or retrieval.

### **Structured handoff**

A parent should not treat a child agent's narrative as authoritative. The parent should validate the returned evidence IDs against the canonical evidence registry and check the support relationship.

### **Parent-child context isolation**

The policy researcher might read twenty passages. The parent does not need all twenty passages in its active context.

Instead, the child can return a compact finding with references, and the parent can selectively retrieve exact passages when necessary.

Delegation therefore performs both task separation and context isolation.

Anthropic's multi-agent research engineering article discusses this pattern, while noting that coordination and evaluation introduce additional complexity.

## Parallel execution and bounded concurrency

Concurrency can improve elapsed time when tasks are independent.

It also increases simultaneous resource usage.

Here is a framework-independent educational dispatcher.

### **Code walkthrough**

`asyncio.Semaphore` limits simultaneous operations.

Think of it as a concurrency limiter around `Promise.all()` in TypeScript.

`asyncio.gather()` waits for all requested tasks here, while each task converts supported failures into a result.

`asyncio.timeout()` applies a per-task timeout. It requests cancellation of an overdue coroutine, but it does not guarantee that blocking external work or remote operations stop immediately.

The dispatcher in `planning.py` also enforces a total deadline, a maximum task count and cancellation cleanup while preserving successful sibling results.

This code demonstrates **parallel execution inside one process**. It is not durable background execution.

A process crash can still lose in-flight tasks.

## When delegation is worse

For the question:

> Who approves a production restore?

A delegated approach might involve a parent model invocation, a child invocation, a child search, and a parent verification step.

A direct RAG approach needs one search and one answer-generation step.

Delegation provides little value in that scenario.

A useful architecture experiment compares the single-agent and delegated designs on the same cases.

Collect task success, claim support, number of retrieved passages, latency, model invocations, and resource consumption.

Do not assume that more agents produce better answers.

## Mental model

Read the following diagram as an application boundary: the model proposes actions while the runtime performs validated operations. The figure renders through Matplotlib in the notebook output.

In [ ]:
display(diagram('Planning and delegation', ['Research restore', 'Research incident', 'Research access', 'Verify handoffs', 'Compare authorities', 'Draft + human review'], [(0, 3), (1, 3), (2, 3), (3, 4), (4, 5)]))
plt.close('all')

## Implementation from scratch

Build the smallest inspectable mechanism before connecting it to the full application. The code below exposes state and checks the invariant that motivates this module.

In [ ]:
from dataclasses import dataclass,field

@dataclass
class LearnerTask:
    id: str
    dependencies: set[str] = field(default_factory=set)
    required_evidence: set[str] = field(default_factory=set)
    completed: bool = False

def validate_dependencies(tasks):
    visited,active = set(),set()
    def visit(key):
        if key not in tasks:
            raise ValueError("Unknown dependency")
        if key in active:
            raise ValueError("Dependency cycle")
        if key in visited:
            return
        active.add(key)
        for parent in tasks[key].dependencies:
            visit(parent)
        active.remove(key)
        visited.add(key)
    for key in tasks:
        visit(key)

def learner_ready(tasks):
    validate_dependencies(tasks)
    return [key for key,t in tasks.items() if not t.completed
            and all(tasks[parent].completed for parent in t.dependencies)]

def learner_complete(tasks,key,evidence):
    task = tasks[key]
    if key not in learner_ready(tasks) or not evidence or not task.required_evidence <= evidence:
        raise ValueError("Completion criterion lacks evidence")
    task.completed = True

async def learner_dispatch(tasks,perform,limit=5):
    for _ in range(limit):
        ready = learner_ready(tasks)
        if not ready:
            break
        key = ready[0]
        learner_complete(tasks,key,await perform(key))
    return tasks

for broken in [{"a":LearnerTask("a",{"unknown"})},
               {"a":LearnerTask("a",{"b"}),"b":LearnerTask("b",{"a"})}]:
    try:
        learner_ready(broken)
    except ValueError as error:
        print("Rejected invalid plan",error)
learner_plan = {"read":LearnerTask("read",required_evidence={"source"})}
try:
    learner_complete(learner_plan,"read",set())
except ValueError as error:
    print("Rejected unsupported completion",error)
# Repair by reading a real source, then bind the completion criterion to its actual ID.
from deep_agent_lab.knowledge import local_search
actual = local_search(ROOT / "data/sample","production restore",1)
learner_plan["read"].required_evidence = {actual[0].id}
learner_complete(learner_plan,"read",{actual[0].id})
assert learner_plan["read"].completed

import asyncio
learner_plan["read"].completed = False
async def read_evidence(key):
    rows = await asyncio.to_thread(local_search,ROOT/"data/sample","production restore",1)
    return {row.id for row in rows}
await learner_dispatch(learner_plan,read_evidence)
assert learner_plan["read"].completed


In [ ]:
from deep_agent_lab.planning import PlanTask,complete_task,ready_tasks,SpecialistFinding,run_specialists
from deep_agent_lab.knowledge import local_search
restore_evidence = local_search(ROOT / "data/sample","production restore",1)[0]
incident_evidence = local_search(ROOT / "data/sample","incident commander",1)[0]
actual_registry = {restore_evidence.id,incident_evidence.id}
tasks = {"restore":PlanTask("restore","Find restoration authority"),
         "incident":PlanTask("incident","Find incident coordinator"),
         "compare":PlanTask("compare","Compare authorities",{"restore","incident"})}
assert {task.id for task in ready_tasks(tasks)} == {"restore","incident"}
complete_task(tasks,"restore",{restore_evidence.id},actual_registry)
complete_task(tasks,"incident",{incident_evidence.id},actual_registry)
assert [task.id for task in ready_tasks(tasks)] == ["compare"]
finding = SpecialistFinding(question="Who approves?",claim="Proposed source finding",
    evidence_ids=[restore_evidence.id],quotes=[restore_evidence.excerpt],
    revisions=[restore_evidence.revision],status="proposed")
print(finding.model_dump())

from deep_agent_lab.planning import verify_handoff
from deep_agent_lab.evidence import source_sentences
complete_finding = finding.model_copy(update={"claim":source_sentences(restore_evidence.excerpt)[0]})
incomplete_finding = complete_finding.model_copy(update={"quotes":[]})
registry = {restore_evidence.id:restore_evidence}
assert verify_handoff(incomplete_finding,registry).label == "insufficient"
assert verify_handoff(complete_finding,registry).label == "supported"
print("Omitted handoff quote rejected; exact source-backed handoff accepted.")


## Framework implementation

The following cell executes the existing Deep Agents/LangGraph application or its actual integration boundary. Inspect the actual service configuration and returned observations in each cell.

In [ ]:
from deep_agent_lab.labs import delegation_lab
delegation = await delegation_lab()
observed_text = "\n".join(delegation["results"]+[delegation["answer"]]).casefold()
assert "database operations lead" in observed_text and "incident commander" in observed_text
assert "task" in delegation["steps"]
print(delegation)


## Code walkthrough

The task graph has two independent research nodes and a dependent comparison. `complete_task` accepts only evidence present in the trusted registry. A todo label is coordination metadata and is insufficient for verification. The existing Deep Agents lab uses separate child contexts and real task tools; child models execute actual research calls and their handoffs require verification. `asyncio.Semaphore` is comparable to a concurrency limiter around Promise.all. Our dispatcher adds per-task timeouts, a total deadline and cancellation cleanup; it remains process-local.

## Interactive experiments

Change concurrency from one to two for independent I/O-bound tasks. Compare elapsed time while holding the simulated work constant. These timings demonstrate scheduling and are not model benchmarks.

In [ ]:
import asyncio,time
from deep_agent_lab.model_lab import course_model
from deep_agent_lab.knowledge import search_knowledge
from langchain.messages import HumanMessage
async def research(topic):
    evidence = await asyncio.to_thread(search_knowledge,settings,topic)
    reply = await course_model().ainvoke([HumanMessage("Use these passages to answer the question. "
        +topic+"\n"+"\n".join(row.excerpt for row in evidence))])
    return {"topic":topic,"evidence_ids":[row.id for row in evidence],"answer":reply.text,
            "usage":reply.usage_metadata}
elapsed = {}
for concurrency in [1,2]:
    started = time.perf_counter()
    findings = await run_specialists(["production restore approval","SEV-1 coordinator"],research,
                                    concurrency=concurrency,timeout=90,deadline=180)
    elapsed[str(concurrency)] = (time.perf_counter()-started)*1000
    print(concurrency,findings)
display(bars("Actual specialist execution",elapsed,measurement_kind="live measurement",
             ylabel="Elapsed milliseconds"))
plt.close("all")

overlapping_requests = []
async def overlapping_research(topic):
    query = "production restore"
    overlapping_requests.append(query)
    evidence = await asyncio.to_thread(search_knowledge,settings,query)
    reply = await course_model().ainvoke([HumanMessage(topic+"\n"+"\n".join(row.excerpt for row in evidence))])
    return {"evidence_ids":[row.id for row in evidence],"answer":reply.text,"usage":reply.usage_metadata}
overlapped = await run_specialists(["restore approval","restore verification"],overlapping_research,
                                  concurrency=2,timeout=90,deadline=180)
print("Actual overlapping specialist requests:",overlapping_requests,overlapped)
print("Duplicate query count:",len(overlapping_requests)-len(set(overlapping_requests)))


## Failure experiment

Deliberately violate a contract or remove a required condition. A caught, asserted failure is an expected result; an uncaught error means this notebook has not passed.

In [ ]:
# A real provider call gets a deliberately short deadline. Sibling work keeps its own budget.
async def deadline_research(topic):
    if topic == "short-deadline":
        async with asyncio.timeout(0.000001):
            return await research("production restore approval")
    return await research("SEV-1 incident coordinator")
partial = await run_specialists(["short-deadline","sibling"],deadline_research,
                               timeout=90,deadline=180)
print(partial)
assert set(partial) == {"short-deadline","sibling"}


## Verification

These assertions inspect actual state produced above. They establish the stated contracts within this lab's scope.

In [ ]:
assert tasks["compare"].status == "pending"
try:
    complete_task(tasks, "compare", {"invented"}, actual_registry)
except ValueError:
    print("Expected: child assertions cannot verify nonexistent evidence.")
else:
    raise AssertionError("Unverified handoff accepted")
from deep_agent_lab.courseware import dependencies
display(dependencies({key:task.dependencies for key,task in tasks.items()}))
plt.close("all")


## Exercise

Create a new plan for a partial-failure run. Verify only the restore task and show that comparison stays blocked until incident evidence is independently verified.

Try your implementation before running the complete reference solution in the next cell.

### Reference solution

Compare the implementation and assertions with your attempt.

In [ ]:
partial_plan = {"restore":PlanTask("restore","research"),
                "incident":PlanTask("incident","research"),
                "compare":PlanTask("compare","compare",{"restore","incident"})}
complete_task(partial_plan,"restore",{restore_evidence.id},actual_registry)
assert [task.id for task in ready_tasks(partial_plan)] == ["incident"]
assert partial_plan["compare"].status == "pending"


## Trade-offs and further reading

Delegation can isolate large contexts and shorten independent work. It also adds parent/child model calls and verification. For one short passage, direct RAG is usually the simpler baseline. Async execution alone cannot survive a process crash.

[Primary source 1](https://docs.langchain.com/oss/python/deepagents/subagents) Â· [Primary source 2](https://www.anthropic.com/engineering/multi-agent-research-system)

## Prediction, intermediate state and repair
The learner dispatcher advances only ready tasks whose acceptance evidence exists.
Inspect the rejected cycles, missing dependencies and empty completion evidence. Repair
the plan using the actual source ID before comparing it with the framework helpers.

## What this lab proves
These assertions verify dependency transitions and bounded scheduling. The specialist
experiment records actual retrieval, model calls and observed partial failure.

## What this lab does not prove
A lower duration does not establish better answers. Synthetic sleeps demonstrate scheduling only. The current experiment uses actual provider calls and must also
be inspected for duplicated retrieval, missing handoff evidence and coordination overhead.


In [ ]:
plt.close('all')
scratch.cleanup()
print('All executed module assertions passed.')